# Census Income API Exploration


## 1. Setup


In [1]:
# os → lets Python interact with things from your operating-system environment
# requests → lets Python communicate with the Census API over HTTP
# load_dotenv → reads the variables stored inside our .env file
import os
import requests
import pandas as pd 

from dotenv import load_dotenv

In [2]:
# This reads the .env file and loads the variables inside it into the environment.
load_dotenv()
# Save CENSUS_API_KEY as census_api_key
census_api_key = os.getenv("CENSUS_API_KEY")
print(census_api_key is not None)

True


## 2. Extract Data



In [3]:
url = "https://api.census.gov/data/2024/acs/acs5"
# params is like dictionary:
params = {
    "get": "NAME,B19013_001E",
    "for": "metropolitan statistical area/micropolitan statistical area:*",
    "key": census_api_key
}

response = requests.get(url, params=params)
response.raise_for_status()
print(response.status_code)

200


In [4]:
data = response.json()
print(type(data))
print(data[:5])

<class 'list'>
[['NAME', 'B19013_001E', 'metropolitan statistical area/micropolitan statistical area'], ['Aberdeen, SD Micro Area', '71488', '10100'], ['Aberdeen, WA Micro Area', '64414', '10140'], ['Abilene, TX Metro Area', '67328', '10180'], ['Ada, OK Micro Area', '63017', '10220']]


## 3. Transform Data


In [5]:
df_income = pd.DataFrame(data[1:],columns = data[0])
df_income.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 935 entries, 0 to 934
Data columns (total 3 columns):
 #   Column                                                       Non-Null Count  Dtype 
---  ------                                                       --------------  ----- 
 0   NAME                                                         935 non-null    object
 1   B19013_001E                                                  935 non-null    object
 2   metropolitan statistical area/micropolitan statistical area  935 non-null    object
dtypes: object(3)
memory usage: 22.0+ KB


In [6]:
df_income.head(10)

,NAME,B19013_001E,metropolitan statistical area/micropolitan statistical area
0,"Aberdeen, SD Micro Area",71488,10100
1,"Aberdeen, WA Micro Area",64414,10140
2,"Abilene, TX Metro Area",67328,10180
3,"Ada, OK Micro Area",63017,10220
4,"Adrian, MI Micro Area",70518,10300
5,"Aguadilla, PR Metro Area",22398,10380
6,"Akron, OH Metro Area",72371,10420
7,"Alamogordo, NM Micro Area",55876,10460
8,"Alamosa, CO Micro Area",51237,10480
9,"Albany, GA Metro Area",56906,10500


In [7]:
df_income = df_income.rename(columns={
    "NAME": "metro_name",
    "B19013_001E": "median_household_income",
    "metropolitan statistical area/micropolitan statistical area": "metro_code"
})

In [8]:
df_income.head()

,metro_name,median_household_income,metro_code
0,"Aberdeen, SD Micro Area",71488,10100
1,"Aberdeen, WA Micro Area",64414,10140
2,"Abilene, TX Metro Area",67328,10180
3,"Ada, OK Micro Area",63017,10220
4,"Adrian, MI Micro Area",70518,10300


In [9]:
df_income["median_household_income"] = pd.to_numeric(
    df_income["median_household_income"],
    errors="coerce"
)

In [10]:
df_income.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 935 entries, 0 to 934
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   metro_name               935 non-null    object
 1   median_household_income  935 non-null    int64 
 2   metro_code               935 non-null    object
dtypes: int64(1), object(2)
memory usage: 22.0+ KB


In [11]:
df_income.isnull().sum()

metro_name                 0
median_household_income    0
metro_code                 0
dtype: int64

In [12]:
df_income.describe()

,median_household_income
count,935.000000
mean,68624.634225
std,15488.458927
min,18917.000000
25%,59424.500000
50%,66806.000000
75%,75948.000000
max,162111.000000


In [13]:
df_income["metro_name"].head(20)

0                    Aberdeen, SD Micro Area
1                    Aberdeen, WA Micro Area
2                     Abilene, TX Metro Area
3                         Ada, OK Micro Area
4                      Adrian, MI Micro Area
5                   Aguadilla, PR Metro Area
6                       Akron, OH Metro Area
7                  Alamogordo, NM Micro Area
8                     Alamosa, CO Micro Area
9                      Albany, GA Metro Area
10                     Albany, OR Metro Area
11    Albany-Schenectady-Troy, NY Metro Area
12                  Albemarle, NC Micro Area
13                 Albert Lea, MN Micro Area
14                Albertville, AL Micro Area
15                Albuquerque, NM Metro Area
16             Alexander City, AL Micro Area
17                 Alexandria, LA Metro Area
18                 Alexandria, MN Micro Area
19                      Alice, TX Micro Area
Name: metro_name, dtype: object

In [14]:
df_income["area_type"] = df_income["metro_name"].apply(
    lambda x: "Metro" if "Metro Area" in x else "Micro"
)

In [15]:
df_income["area_type"].value_counts()

area_type
Micro    542
Metro    393
Name: count, dtype: int64

In [16]:
df_metro_income = df_income[df_income["area_type"] == "Metro"].copy()

In [17]:
df_metro_income.head()

,metro_name,median_household_income,metro_code,area_type
2,"Abilene, TX Metro Area",67328,10180,Metro
5,"Aguadilla, PR Metro Area",22398,10380,Metro
6,"Akron, OH Metro Area",72371,10420,Metro
9,"Albany, GA Metro Area",56906,10500,Metro
10,"Albany, OR Metro Area",76329,10540,Metro


In [18]:
df_metro_income[
    df_metro_income["metro_name"].str.contains("New York", case=False)
]

,metro_name,median_household_income,metro_code,area_type
607,"New York-Newark-Jersey City, NY-NJ Metro Area",99155,35620,Metro


In [19]:
df_metro_income[
    df_metro_income["metro_name"].str.contains(
        "Dallas|Chicago|Miami",
        case=False,
        regex=True
    )
][["metro_name", "metro_code", "median_household_income"]]

,metro_name,metro_code,median_household_income
162,"Chicago-Naperville-Elgin, IL-IN Metro Area",16980,90887
210,"Dallas-Fort Worth-Arlington, TX Metro Area",19100,90275
545,"Miami-Fort Lauderdale-West Palm Beach, FL Metr...",33100,76527


In [20]:
print(df_metro_income.isnull().sum())
print(df_metro_income.describe())

metro_name                 0
median_household_income    0
metro_code                 0
area_type                  0
dtype: int64
       median_household_income
count               393.000000
mean              74545.577608
std               15733.238661
min               21163.000000
25%               65158.000000
50%               72475.000000
75%               81861.000000
max              162111.000000


## 4. Data Validation


In [21]:
# assert is to raise an error if the condition is not True
assert df_metro_income["metro_code"].isnull().sum() == 0
assert df_metro_income["metro_name"].isnull().sum() == 0
assert df_metro_income["median_household_income"].isnull().sum() == 0
# Every metro_code must occur once
assert df_metro_income["metro_code"].is_unique
# Every income must be greater than zero.
assert (df_metro_income["median_household_income"] > 0).all()

## 5. Load Data

In [22]:
import json
with open("../data/raw/census_income_2024.json", "w") as file:
    json.dump(data, file, indent=4)

In [23]:
df_metro_income["metro_name"] = (
    df_metro_income["metro_name"]
    .str.replace(" Metro Area", "", regex=False)
)

In [24]:
df_metro_income = df_metro_income.drop(columns=["area_type"])

In [25]:
df_metro_income = df_metro_income[
    [
        "metro_code",
        "metro_name",
        "median_household_income"
    ]
]

In [26]:
df_metro_income.to_csv(
    "../data/processed/census_metro_income_2024.csv",
    index=False
)

In [27]:
df=pd.read_csv("../data/processed/redfin_monthly_housing.csv")

In [28]:
df["redfin_region_id"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 129578 entries, 0 to 129577
Series name: redfin_region_id
Non-Null Count   Dtype
--------------   -----
129578 non-null  int64
dtypes: int64(1)
memory usage: 1012.5 KB


In [29]:
df_income["metro_code"]=df_income["metro_code"].astype("int64")
df_income["metro_code"].info()

<class 'pandas.core.series.Series'>
RangeIndex: 935 entries, 0 to 934
Series name: metro_code
Non-Null Count  Dtype
--------------  -----
935 non-null    int64
dtypes: int64(1)
memory usage: 7.4 KB


In [30]:
df_census = pd.read_csv(
    "../data/processed/census_metro_income_2024.csv",
    dtype={"metro_code": str}
)

census_metro_ids = set(df_census["metro_code"])

In [31]:
redfin_ids = set(df["redfin_region_id"].astype(str))
census_metro_ids = set(df_metro_income["metro_code"].astype(str))

direct_matches = redfin_ids & census_metro_ids
unmatched_redfin_ids = redfin_ids - census_metro_ids

print("Redfin IDs:", len(redfin_ids))
print("Census metro IDs:", len(census_metro_ids))
print("Direct metro matches:", len(direct_matches))
print("Unmatched Redfin IDs:", len(unmatched_redfin_ids))

Redfin IDs: 946
Census metro IDs: 393
Direct metro matches: 368
Unmatched Redfin IDs: 578


In [32]:
census_micro_ids = set(df_income[df_income["area_type"] == "Micro"]["metro_code"].astype(str))
micro_matches = unmatched_redfin_ids & census_micro_ids

print("Redfin IDs matching Census Micro areas:", len(micro_matches))

unmatched_after_micro = unmatched_redfin_ids - census_micro_ids

print("Still unmatched after removing Micro areas:", len(unmatched_after_micro))

Redfin IDs matching Census Micro areas: 494
Still unmatched after removing Micro areas: 84


In [33]:
division_parent_metros = [
    "31080",  # Los Angeles
    "33100",  # Miami
    "19820",  # Detroit
    "47900",  # Washington
    "41860",  # San Francisco
    "19100",  # Dallas
    "14460",  # Boston
    "16980",  # Chicago
    "37980",  # Philadelphia
    "35620",  # New York
    "42660"   # Seattle
]

In [34]:
url = "https://api.census.gov/data/2024/acs/acs5"

all_divisions = []
division_columns = None

for metro_code in division_parent_metros:

    params = {
        "get": "NAME,B19013_001E",
        "for": "metropolitan division:*",
        "in": f"metropolitan statistical area/micropolitan statistical area:{metro_code}",
        "key": census_api_key
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    division_data = response.json()

    if division_columns is None:
        division_columns = division_data[0]

    all_divisions.extend(division_data[1:])

In [35]:
df_divisions = pd.DataFrame(all_divisions,columns=division_columns)

In [36]:
print(df_divisions.shape)
df_divisions.head()

(33, 4)


,NAME,B19013_001E,metropolitan statistical area/micropolitan statistical area,metropolitan division
0,"Anaheim-Santa Ana-Irvine, CA Metro Division; L...",116289,31080,11244
1,"Los Angeles-Long Beach-Glendale, CA Metro Divi...",90112,31080,31084
2,"Fort Lauderdale-Pompano Beach-Sunrise, FL Metr...",77633,33100,22744
3,"Miami-Miami Beach-Kendall, FL Metro Division; ...",71753,33100,33124
4,"West Palm Beach-Boca Raton-Delray Beach, FL Me...",83581,33100,48424


In [37]:
division_ids = set(df_divisions["metropolitan division"].astype(str))

division_matches= division_ids & unmatched_after_micro

still_unmatched = unmatched_after_micro - division_ids

print("Division matches",len(division_matches))
print("Still unmatched",len(still_unmatched))

Division matches 27
Still unmatched 57


In [38]:
print(df["redfin_region_id"].dtype)
print(type(next(iter(still_unmatched))))

int64
<class 'str'>


In [39]:
df_still_unmatched = (
    df[
        df["redfin_region_id"].astype(str).isin(still_unmatched)
    ][["redfin_region_id", "region_name"]]
    .drop_duplicates()
    .sort_values("region_name")
)

df_still_unmatched

,redfin_region_id,region_name
5386,11780,"Ashtabula, OH metro area"
6591,12120,"Atmore, AL metro area"
8398,12680,"Bardstown, KY metro area"
11178,13500,"Bennettsville, SC metro area"
11456,13620,"Berlin, NH metro area"
11801,13720,"Big Stone Gap, VA metro area"
13329,14160,"Bluffton, IN metro area"
16330,15140,"Brownsville, TN metro area"
18137,15680,"California, MD metro area"
20760,16420,"Central City, KY metro area"


Census → income

FRED → mortgage rates

Redfin → home prices / market metrics

Zillow → rent

Geography bridge → connects Zillow ↔ Redfin ↔ Census